# LLM-FFT Regression Analysis

Section 4.4 of the paper: OLS regression of community-level voted policy characteristics
on demographic/socioeconomic variables.

**Outcome variables** (Borda-weighted mean across 10 rounds per community):
- `borda_tax`  — mean weighted tax rate voted for
- `borda_fare` — mean weighted fare voted for
- `borda_fee`  — mean weighted driver fee voted for

**Predictors** (from `data/CA_info/`):
- Household size, race composition, car ownership, transit use, income distribution

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
import ast, warnings
warnings.filterwarnings('ignore')

RESULTS_ROOT = Path('./results')
CA_INFO_DIR  = Path('./data/CA_info')
POLICY_CSV   = Path('./data/policy_chicago.csv')
FIG_DIR = Path('./figures'); FIG_DIR.mkdir(exist_ok=True)
NUM_ROUNDS = 10
print('Setup complete.')

## 1. Load & Prepare Data

In [ ]:
def sanitize_name(name):
    name = name.replace('/', '&')
    name = re.sub(r"[^\w\s&-]", '', name)
    return name.replace(' ', '_').lower()

def parse_vote_list(v):
    if isinstance(v, list): return v
    try: return ast.literal_eval(str(v))
    except: return []


def load_policy_params(policy_csv):
    df = pd.read_csv(policy_csv)
    return df.set_index('index') if 'index' in df.columns else df


def load_ca_info(ca_info_dir):
    records = []
    for jf in sorted(Path(ca_info_dir).glob('*.json')):
        try:
            d = json.loads(jf.read_text(encoding='utf-8'))
            row = {'community_file': jf.stem}
            for cat, vals in d.items():
                for k, v in vals.items(): row[f'{cat}: {k}'] = v
            records.append(row)
        except: pass
    return pd.DataFrame(records)


# Load
policy_df = load_policy_params(POLICY_CSV)
ca_raw = load_ca_info(CA_INFO_DIR)
print(f'Policy rows: {len(policy_df)}, CA communities: {len(ca_raw)}')

In [ ]:
# Normalize demographic data
ca_demo = ca_raw.rename(columns={
    'Population: Total Population':             'pop_total',
    'Population: Total Households':             'pop_households',
    'Population: Average Household Size':       'household_size',
    'Race: White (Non-Hispanic)':               'race_white',
    'Race: Hispanic or Latino (Any Race)':      'race_hispanic',
    'Race: Black (Non-Hispanic)':               'race_black',
    'Race: Asian (Non-Hispanic)':               'race_asian',
    'Race: Other/Multiple Races (Non-Hispanic)':'race_other',
    'Car Ownership: No Vehicle':                'car_no',
    'Car Ownership: 1 Vehicle':                 'car_1',
    'Car Ownership: 2 Vehicles':                'car_2',
    'Car Ownership: 3+ Vehicles':               'car_3plus',
    'Travel Mode: Work at Home':                'tvl_wfh',
    'Travel Mode: Drive Alone':                 'tvl_drive',
    'Travel Mode: Carpool':                     'tvl_carpool',
    'Travel Mode: Transit':                     'tvl_transit',
    'Travel Mode: Walk or Bike':                'tvl_walk_bike',
    'Travel Mode: Other':                       'tvl_other',
    'Income: Less than $25,000':                'income_lt25k',
    'Income: $25,000 to $49,999':               'income_25_49k',
    'Income: $50,000 to $74,999':               'income_50_74k',
    'Income: $75,000 to $99,999':               'income_75_99k',
    'Income: $100,000 to $149,999':             'income_100_149k',
    'Income: $150,000 and Over':                'income_150k_plus',
    'Income: Median Income':                    'median_income',
    'Income: Per Capita Income':                'per_capita_income',
})

pct_cols = [c for c in ca_demo.columns if any(c.startswith(p) for p in
    ['race_','car_','tvl_','income_lt','income_25','income_50','income_75','income_100','income_150'])]
ca_demo[pct_cols] = ca_demo[pct_cols] / 100.0

# Derived
ca_demo['pct_non_white']  = 1.0 - ca_demo.get('race_white', pd.Series(dtype=float))
ca_demo['pct_no_vehicle'] = ca_demo.get('car_no', pd.Series(dtype=float))

print(f'Demographic columns: {ca_demo.shape}')
ca_demo.head(2)

In [ ]:
# Descriptive statistics for key variables
key_vars = ['household_size','race_white','race_black','race_hispanic','race_asian',
            'car_no','tvl_transit','tvl_drive','income_lt25k','income_150k_plus',
            'median_income','pct_non_white']
desc = ca_demo[[v for v in key_vars if v in ca_demo.columns]].describe().T
print('Descriptive statistics of community demographics:')
display(desc.round(4))

## 2. Compute Borda-Weighted Mean Voted Policy Values

In [ ]:
def compute_borda_means(exp_dir, policy_df, num_rounds=NUM_ROUNDS, weights=(5,4,3,2,1)):
    comm_data = {}
    for i in range(1, num_rounds+1):
        csv_path = Path(exp_dir) / f'round_{i}' / 'combined_voting_results.csv'
        if not csv_path.exists(): continue
        df = pd.read_csv(csv_path)
        rank_cols = [f'rank{j}' for j in range(1, len(weights)+1) if f'rank{j}' in df.columns]
        for _, row in df.iterrows():
            ca = row['community_area']
            if ca not in comm_data:
                comm_data[ca] = dict(w_tax=0.0, w_fare=0.0, w_fee=0.0, w_total=0.0)
            for w, rc in zip(weights, rank_cols):
                val = row.get(rc)
                if pd.isna(val): continue
                pid = int(val)
                if pid not in policy_df.index: continue
                p = policy_df.loc[pid]
                comm_data[ca]['w_tax']   += w * p['tax_percentage']
                comm_data[ca]['w_fare']  += w * p['fare']
                comm_data[ca]['w_fee']   += w * p['driver_fee']
                comm_data[ca]['w_total'] += w
    rows = []
    for ca, d in comm_data.items():
        if d['w_total'] > 0:
            rows.append({'community_area': ca, 'file_stem': sanitize_name(ca),
                         'borda_tax':  d['w_tax']  / d['w_total'],
                         'borda_fare': d['w_fare'] / d['w_total'],
                         'borda_fee':  d['w_fee']  / d['w_total']})
    return pd.DataFrame(rows)


# Compute for all four Chicago ranked experiments
experiments = {
    'GPT-4o':        RESULTS_ROOT / 'chicago_gpt_ranked',
    'GPT-4o (info)': RESULTS_ROOT / 'chicago_gpt_ranked_info',
    'Claude':        RESULTS_ROOT / 'chicago_claude_ranked',
    'Claude (info)': RESULTS_ROOT / 'chicago_claude_ranked_info',
}

borda_dfs = {}
for label, exp_dir in experiments.items():
    if not exp_dir.exists(): print(f'Skip: {exp_dir}'); continue
    bm = compute_borda_means(exp_dir, policy_df)
    borda_dfs[label] = bm
    print(f'{label}: {len(bm)} communities, mean tax={bm["borda_tax"].mean():.3f}')

In [ ]:
# Merge borda means with demographics
merged_dfs = {}
for label, bm in borda_dfs.items():
    merged = pd.merge(bm, ca_demo, left_on='file_stem', right_on='community_file', how='inner')
    # Rename borda columns to include model label
    safe = label.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    merged = merged.rename(columns={
        'borda_tax':  f'borda_tax_{safe}',
        'borda_fare': f'borda_fare_{safe}',
        'borda_fee':  f'borda_fee_{safe}',
    })
    merged_dfs[label] = merged
    print(f'{label}: {len(merged)} rows after merge.')

## 3. VIF Analysis (Multicollinearity Check)

In [ ]:
# Check VIF for the key predictor set
X_ALL = ['household_size','pct_non_white','race_black','race_hispanic','race_asian',
         'car_no','tvl_transit','tvl_drive','income_lt25k','income_150k_plus']

if merged_dfs:
    sample_df = list(merged_dfs.values())[0]
    x_avail = [c for c in X_ALL if c in sample_df.columns]
    sub = sample_df[x_avail].dropna()
    X_mat = sm.add_constant(sub.astype(float))
    vif_data = pd.DataFrame({
        'Variable': X_mat.columns,
        'VIF':      [variance_inflation_factor(X_mat.values, i) for i in range(X_mat.shape[1])]
    })
    print('VIF (>10 suggests multicollinearity):')
    display(vif_data.sort_values('VIF', ascending=False).round(2))

In [ ]:
# Correlation heatmap among predictors
if merged_dfs:
    sample_df = list(merged_dfs.values())[0]
    x_avail = [c for c in X_ALL if c in sample_df.columns]
    corr = sample_df[x_avail].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                linewidths=0.3, ax=ax, annot_kws={'size': 8})
    ax.set_title('Predictor Correlation Matrix', fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'reg_corr_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. OLS Regression — Parsimonious Model (Paper Table 4)

In [ ]:
def run_ols(data, x_cols, y_col, standardize=True):
    sub = data[x_cols + [y_col]].dropna()
    if sub.empty: return None
    X = sub[x_cols].astype(float)
    y = sub[y_col].astype(float)
    if standardize:
        X = pd.DataFrame(StandardScaler().fit_transform(X), columns=x_cols, index=X.index)
    return sm.OLS(y, sm.add_constant(X)).fit()


# Parsimonious predictor set (low-VIF subset)
X_PARS = ['household_size', 'pct_non_white', 'car_no', 'tvl_transit', 'income_lt25k', 'income_150k_plus']

print('Parsimonious predictor set:', X_PARS)

In [ ]:
# Regression results for all experiments
all_models = {}   # {(label, lever): model}

for label, merged in merged_dfs.items():
    safe = label.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    all_models[label] = {}
    x_avail = [c for c in X_PARS if c in merged.columns]

    print(f'\n{"="*65}')
    print(f'  Experiment: {label}')
    print(f'{"="*65}')

    for lever_name, col_suffix in [('Tax rate','borda_tax'), ('Fare','borda_fare'), ('Driver fee','borda_fee')]:
        y_col = f'{col_suffix}_{safe}'
        if y_col not in merged.columns:
            print(f'  Column {y_col} not found — skip.')
            continue
        model = run_ols(merged, x_avail, y_col)
        all_models[label][lever_name] = model
        if model:
            print(f'\n  Y = {lever_name} | n={int(model.nobs)}, R²={model.rsquared:.3f}, adj-R²={model.rsquared_adj:.3f}, F-p={model.f_pvalue:.4f}')
            tbl = model.summary2().tables[1][['Coef.','Std.Err.','t','P>|t|']].drop('const', errors='ignore')
            display(tbl.style.applymap(lambda v: 'font-weight: bold' if isinstance(v, float) and v < 0.05 else '', subset=['P>|t|']))

In [ ]:
# Summary regression table: R² for all experiment × lever combinations
r2_data = []
for label, lever_models in all_models.items():
    row = {'Experiment': label}
    for lever, model in lever_models.items():
        row[f'R² ({lever})'] = round(model.rsquared, 3) if model else np.nan
        row[f'adj-R² ({lever})'] = round(model.rsquared_adj, 3) if model else np.nan
    r2_data.append(row)

if r2_data:
    print('R² summary across experiments:')
    display(pd.DataFrame(r2_data))

## 5. Coefficient Comparison Plot

In [ ]:
def plot_coefficients(all_models, lever, x_cols, save_path=None):
    """Side-by-side coefficient bar chart for all experiments."""
    n = len(all_models)
    if n == 0: return

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(x_cols))
    bw = 0.8 / n
    colors = sns.color_palette('Set2', n)

    for i, (label, lever_models) in enumerate(all_models.items()):
        model = lever_models.get(lever)
        if not model: continue
        coef = model.params.drop('const', errors='ignore')
        ci   = model.conf_int().drop('const', errors='ignore')
        pvals= model.pvalues.drop('const', errors='ignore')
        vals  = np.array([coef.get(c, np.nan) for c in x_cols])
        lo    = np.array([ci.loc[c, 0] if c in ci.index else np.nan for c in x_cols])
        hi    = np.array([ci.loc[c, 1] if c in ci.index else np.nan for c in x_cols])
        sig   = np.array([pvals.get(c, 1.0) < 0.05 for c in x_cols])

        offset = (i - n/2 + 0.5) * bw
        bars = ax.bar(x + offset, vals, bw,
                      yerr=[vals - lo, hi - vals],
                      capsize=3, color=colors[i], alpha=0.8,
                      label=label, error_kw={'linewidth': 1.2})
        # Mark significant bars with *
        for j, (bar, is_sig) in enumerate(zip(bars, sig)):
            if is_sig and not np.isnan(vals[j]):
                ax.text(bar.get_x() + bar.get_width()/2,
                        vals[j] + (hi[j]-vals[j]) + 0.001,
                        '*', ha='center', va='bottom', fontsize=12, color='black')

    ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(x_cols, rotation=35, ha='right', fontsize=10)
    ax.set_ylabel('Standardized coefficient (95% CI)', fontsize=11)
    ax.set_title(f'Regression coefficients — Y = {lever}\n* = p < 0.05', fontsize=12)
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


if all_models and merged_dfs:
    sample = list(merged_dfs.values())[0]
    x_avail = [c for c in X_PARS if c in sample.columns]
    for lever in ['Tax rate', 'Fare', 'Driver fee']:
        plot_coefficients(all_models, lever, x_avail,
                          save_path=FIG_DIR / f'reg_coef_{lever.replace(" ","_")}.png')

## 6. Scatter Plots — Key Relationships

In [ ]:
# Scatter: pct_no_vehicle × borda_tax for each model
if merged_dfs:
    fig, axes = plt.subplots(1, len(merged_dfs), figsize=(5*len(merged_dfs), 4), sharey=True)
    if len(merged_dfs) == 1: axes = [axes]

    for ax, (label, merged) in zip(axes, merged_dfs.items()):
        safe = label.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
        y_col = f'borda_tax_{safe}'
        if 'car_no' not in merged.columns or y_col not in merged.columns:
            ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
            ax.set_title(label); continue

        ax.scatter(merged['car_no'], merged[y_col], alpha=0.6, s=40)
        # Trend line
        sub = merged[['car_no', y_col]].dropna()
        if len(sub) > 2:
            z = np.polyfit(sub['car_no'], sub[y_col], 1)
            p = np.poly1d(z)
            xs = np.linspace(sub['car_no'].min(), sub['car_no'].max(), 50)
            ax.plot(xs, p(xs), 'r--', linewidth=1.5)
        ax.set_xlabel('% No vehicle', fontsize=11)
        ax.set_ylabel('Borda mean tax rate', fontsize=11)
        ax.set_title(label, fontsize=11)

    plt.suptitle('Vehicle ownership vs. voted tax rate', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'reg_scatter_car_no_tax.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Scatter: tvl_transit × borda_fee
if merged_dfs:
    fig, axes = plt.subplots(1, len(merged_dfs), figsize=(5*len(merged_dfs), 4), sharey=True)
    if len(merged_dfs) == 1: axes = [axes]

    for ax, (label, merged) in zip(axes, merged_dfs.items()):
        safe = label.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
        y_col = f'borda_fee_{safe}'
        if 'tvl_transit' not in merged.columns or y_col not in merged.columns:
            ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
            ax.set_title(label); continue

        ax.scatter(merged['tvl_transit'], merged[y_col], alpha=0.6, s=40, color='steelblue')
        sub = merged[['tvl_transit', y_col]].dropna()
        if len(sub) > 2:
            z = np.polyfit(sub['tvl_transit'], sub[y_col], 1)
            xs = np.linspace(sub['tvl_transit'].min(), sub['tvl_transit'].max(), 50)
            ax.plot(xs, np.poly1d(z)(xs), 'r--', linewidth=1.5)
        ax.set_xlabel('% Transit commuters', fontsize=11)
        ax.set_ylabel('Borda mean driver fee', fontsize=11)
        ax.set_title(label, fontsize=11)

    plt.suptitle('Transit share vs. voted driver fee', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'reg_scatter_transit_fee.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. GPT-4o vs Claude — Borda Mean Comparison

In [ ]:
# Compare Borda means between GPT-4o and Claude directly
if 'GPT-4o' in borda_dfs and 'Claude' in borda_dfs:
    gpt = borda_dfs['GPT-4o'].set_index('community_area')[['borda_tax','borda_fare','borda_fee']]
    cla = borda_dfs['Claude'].set_index('community_area')[['borda_tax','borda_fare','borda_fee']]
    both = gpt.join(cla, lsuffix='_gpt', rsuffix='_claude', how='inner')

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, lever in zip(axes, ['borda_tax','borda_fare','borda_fee']):
        x_col = f'{lever}_gpt'
        y_col = f'{lever}_claude'
        if x_col not in both.columns or y_col not in both.columns: continue
        ax.scatter(both[x_col], both[y_col], alpha=0.6, s=40)
        lims = [min(both[x_col].min(), both[y_col].min()) - 0.02,
                max(both[x_col].max(), both[y_col].max()) + 0.02]
        ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='y=x')
        corr = both[[x_col, y_col]].dropna().corr().iloc[0,1]
        ax.set_xlabel(f'GPT-4o', fontsize=11); ax.set_ylabel('Claude', fontsize=11)
        ax.set_title(f'{lever.replace("borda_","").title()} (r={corr:.3f})', fontsize=11)
        ax.legend(fontsize=9)

    plt.suptitle('GPT-4o vs Claude — Borda-weighted mean voted lever values', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'reg_gpt_vs_claude_borda.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Need both GPT-4o and Claude results to compare.')